# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [2]:
# ============================================================
# CELL 1 — Environment setup for current Google Colab
# ============================================================

# Check Python version
import sys
print("Python:", sys.version)

# Clone Piper sample generator
!git clone https://github.com/dscripka/piper-sample-generator

# Download Piper voice model
!wget -q -O piper-sample-generator/models/en-us-libritts-high.pt \
https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt

# System dependency for phonemization
!apt-get -qq update
!apt-get -qq -y install espeak-ng

# Python dependencies
!pip install -q espeak-phonemizer
!pip install -q webrtcvad || pip install -q webrtcvad-wheels

# Install openWakeWord
!pip install -q --no-deps openwakeword

# Install required training dependencies compatible with current Colab
!pip install -q \
onnxruntime-gpu \
onnxscript \
scikit-learn \
requests \
scipy \
tqdm \
soundfile \
piper-phonemize-fix \
mutagen==1.47.0 \
torchinfo==1.8.0 \
torchmetrics==1.2.0 \
speechbrain==0.5.14 \
audiomentations==0.33.0 \
torch-audiomentations==0.11.0 \
acoustics==0.2.6 \
pronouncing==0.2.0 \
deep-phonemizer==0.0.19 \
"datasets==2.21.0"

# Download openWakeWord base models
from importlib.metadata import version
import openwakeword.utils as oww_utils

print("openwakeword:", version("openwakeword"))
oww_utils.download_models()

print("\n✅ Cell 1 completed successfully")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Cloning into 'piper-sample-generator'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 75 (delta 21), reused 18 (delta 18), pack-reused 40 (from 1)
Receiving objects: 100% (75/75), 1.01 MiB | 11.64 MiB/s, done.
Resolving deltas: 100% (22/22), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1

embedding_model.tflite: 100%|██████████| 1.33M/1.33M [00:00<00:00, 25.6MiB/s]
embedding_model.onnx: 100%|██████████| 1.33M/1.33M [00:00<00:00, 23.0MiB/s]
melspectrogram.tflite: 100%|██████████| 1.09M/1.09M [00:00<00:00, 16.8MiB/s]
melspectrogram.onnx: 100%|██████████| 1.09M/1.09M [00:00<00:00, 21.5MiB/s]
silero_vad.onnx: 100%|██████████| 1.81M/1.81M [00:00<00:00, 25.7MiB/s]
alexa_v0.1.tflite: 100%|██████████| 855k/855k [00:00<00:00, 13.8MiB/s]
alexa_v0.1.onnx: 100%|██████████| 854k/854k [00:00<00:00, 19.9MiB/s]
hey_mycroft_v0.1.tflite: 100%|██████████| 860k/860k [00:00<00:00, 16.3MiB/s]
hey_mycroft_v0.1.onnx: 100%|██████████| 858k/858k [00:00<00:00, 19.7MiB/s]
hey_jarvis_v0.1.tflite: 100%|██████████| 1.28M/1.28M [00:00<00:00, 24.5MiB/s]
hey_jarvis_v0.1.onnx: 100%|██████████| 1.27M/1.27M [00:00<00:00, 22.0MiB/s]
hey_rhasspy_v0.1.tflite: 100%|██████████| 416k/416k [00:00<00:00, 12.5MiB/s]
hey_rhasspy_v0.1.onnx: 100%|██████████| 204k/204k [00:00<00:00, 8.36MiB/s]
timer_v0.1.tflite: 100%|█


✅ Cell 1 completed successfully


In [1]:
!nvidia-smi


Thu Aug 20 15:16:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Imports

import os
import numpy as np
import torch
import sys
from pathlib import Path
import uuid
import yaml
import datasets
import scipy
from tqdm import tqdm


In [4]:
import torch
import torchaudio

print("Torch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.4.1+cu121
Torchaudio: 2.4.1+cu121
CUDA available: True


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [6]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

Resolving data files:   0%|          | 0/270 [00:00<?, ?it/s]

270it [00:53,  5.07it/s]


In [7]:
# ============================================================
# Cell 4 — Download background noise / negative audio
# ============================================================

import os
import numpy as np
import scipy.io.wavfile
import datasets
from pathlib import Path
from tqdm import tqdm

# ---------------------------
# AudioSet background clips
# ---------------------------

output_dir = "./audioset_16k"
N_AUDIOSET = 2000

os.makedirs(output_dir, exist_ok=True)

existing = len(list(Path(output_dir).glob("*.wav")))

if existing < N_AUDIOSET:
    print(f"Downloading AudioSet clips... ({existing}/{N_AUDIOSET} already present)")

    aset = datasets.load_dataset(
        "agkphysics/AudioSet",
        "balanced",
        split="train",
        streaming=True
    )

    aset = aset.cast_column(
        "audio",
        datasets.Audio(sampling_rate=16000)
    )

    done = existing

    for row in tqdm(aset, total=N_AUDIOSET, desc="AudioSet -> 16k"):

        if done >= N_AUDIOSET:
            break

        stem = str(row.get("video_id") or f"audioset_{done:05d}")
        out_file = os.path.join(output_dir, stem + ".wav")

        # Skip if this file already exists
        if os.path.exists(out_file):
            continue

        audio = row["audio"]["array"]

        scipy.io.wavfile.write(
            out_file,
            16000,
            (audio * 32767).astype(np.int16)
        )

        done += 1

print(
    "AudioSet ready:",
    len(list(Path("./audioset_16k").glob("*.wav"))),
    "clips"
)

# ---------------------------
# We will use AudioSet as the
# background dataset for training.
# ---------------------------


Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/38 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

AudioSet -> 16k: 100%|██████████| 2000/2000 [02:09<00:00, 15.45it/s]

AudioSet ready: 2000 clips


In [8]:
# ============================================================
# Cell 5 — Download pre-computed negative training features
# ============================================================

import os

FEATURES = {
    "openwakeword_features_ACAV100M_2000_hrs_16bit.npy": 17280000128,
    "validation_set_features.npy": 184836608,
}

BASE_URL = (
    "https://huggingface.co/datasets/davidscripka/"
    "openwakeword_features/resolve/main/"
)

for filename, expected_size in FEATURES.items():

    # Skip only if the complete file already exists
    if (
        os.path.exists(filename)
        and os.path.getsize(filename) == expected_size
    ):
        print(f"✓ {filename} already downloaded")
        continue

    print(f"\nDownloading: {filename}")

    # -c means continue/resume if the download was interrupted
    !wget -c "{BASE_URL}{filename}" -O "{filename}"

    actual_size = os.path.getsize(filename)

    assert actual_size == expected_size, (
        f"Download incomplete: {filename}\n"
        f"Expected: {expected_size} bytes\n"
        f"Got: {actual_size} bytes\n"
        "Run this cell again to resume."
    )

    print(f"✓ {filename} downloaded successfully")

print("\n🎉 Cell 5 completed successfully")


Downloading: openwakeword_features_ACAV100M_2000_hrs_16bit.npy
--2026-08-20 15:35:52--  https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
Resolving huggingface.co (huggingface.co)... 3.171.171.104, 3.171.171.128, 3.171.171.65, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.104|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64f3a0b6918ffcc15af6923c/7e1cade4c3fda6a5081158383c8d43c4a3e1e42555150b596b373efddf9b5194?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27openwakeword_features_ACAV100M_2000_hrs_16bit.npy%3B+filename%3D%22openwakeword_features_ACAV100M_2000_hrs_16bit.npy%22%3B&user_id=public&Expires=1787243752&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRmM2EwYjY5MThmZmNjMTVhZjY5MjNjLzdlMWNhZGU0YzNmZGE2YTUwODExNTgzODNjOGQ0M2M0YTNlMWU0M

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [15]:
# Load default YAML config file for training

config = yaml.load(
    open("/content/openwakeword/examples/custom_model.yml", "r").read(),
    yaml.Loader
)

config

{'model_name': 'my_model',
 'target_phrase': ['hey jarvis'],
 'custom_negative_phrases': [],
 'n_samples': 10000,
 'n_samples_val': 2000,
 'tts_batch_size': 50,
 'augmentation_batch_size': 16,
 'piper_sample_generator_path': './piper-sample-generator',
 'output_dir': './my_custom_model',
 'rir_paths': ['./mit_rirs'],
 'background_paths': ['./background_clips'],
 'background_paths_duplication_rate': [1],
 'false_positive_validation_data_path': './validation_set_features.npy',
 'augmentation_rounds': 1,
 'feature_data_files': {'ACAV100M_sample': './openwakeword_features_ACAV100M_2000_hrs_16bit.npy'},
 'batch_n_per_class': {'ACAV100M_sample': 1024,
  'adversarial_negative': 50,
  'positive': 50},
 'model_type': 'dnn',
 'layer_size': 32,
 'steps': 50000,
 'max_negative_weight': 1500,
 'target_false_positives_per_hour': 0.2}

In [11]:
!pwd
!ls -la /content
!find /content/openwakeword -maxdepth 3 -type f | head -100

/content
total 17055624
drwxr-xr-x 1 root root        4096 Aug 20 15:42 .
drwxr-xr-x 1 root root        4096 Aug 20 15:14 ..
drwxr-xr-x 2 root root       69632 Aug 20 15:32 audioset_16k
drwxr-xr-x 4 root root        4096 Aug 10 13:31 .config
drwxr-xr-x 2 root root       20480 Aug 20 15:26 mit_rirs
-rw-r--r-- 1 root root 17280000128 Aug 20 15:42 openwakeword_features_ACAV100M_2000_hrs_16bit.npy
drwxr-xr-x 6 root root        4096 Aug 20 15:18 piper-sample-generator
drwxr-xr-x 1 root root        4096 Aug 10 13:31 sample_data
-rw-r--r-- 1 root root   184836608 Aug 20 15:42 validation_set_features.npy
find: ‘/content/openwakeword’: No such file or directory


In [12]:
!cd /content && git clone https://github.com/dscripka/openWakeWord.git openwakeword


Cloning into 'openwakeword'...
remote: Enumerating objects: 1248, done.
remote: Counting objects: 100% (724/724), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 1248 (delta 605), reused 563 (delta 563), pack-reused 524 (from 1)
Receiving objects: 100% (1248/1248), 3.23 MiB | 23.80 MiB/s, done.
Resolving deltas: 100% (776/776), done.


In [13]:
!find /content/openwakeword -maxdepth 3 -type f | head -50


/content/openwakeword/README.md
/content/openwakeword/.git/index
/content/openwakeword/.git/logs/HEAD
/content/openwakeword/.git/hooks/update.sample
/content/openwakeword/.git/hooks/post-update.sample
/content/openwakeword/.git/hooks/pre-push.sample
/content/openwakeword/.git/hooks/pre-receive.sample
/content/openwakeword/.git/hooks/applypatch-msg.sample
/content/openwakeword/.git/hooks/fsmonitor-watchman.sample
/content/openwakeword/.git/hooks/pre-applypatch.sample
/content/openwakeword/.git/hooks/push-to-checkout.sample
/content/openwakeword/.git/hooks/pre-merge-commit.sample
/content/openwakeword/.git/hooks/pre-rebase.sample
/content/openwakeword/.git/hooks/commit-msg.sample
/content/openwakeword/.git/hooks/prepare-commit-msg.sample
/content/openwakeword/.git/hooks/pre-commit.sample
/content/openwakeword/.git/packed-refs
/content/openwakeword/.git/config
/content/openwakeword/.git/HEAD
/content/openwakeword/.git/info/exclude
/content/openwakeword/.git/description
/content/openwakewo

In [14]:
!find /content/openwakeword -type f \( -name "*.yml" -o -name "*.yaml" \)

/content/openwakeword/.github/workflows/tests.yml
/content/openwakeword/.github/workflows/build_and_publish_to_pypi.yml
/content/openwakeword/examples/custom_model.yml


In [16]:
# Configure and save the custom wake-word model

config["target_phrase"] = ["hey raizen"]
config["model_name"] = "hey_raizen"

# Start with a manageable training size
config["n_samples"] = 1000
config["n_samples_val"] = 1000

# Training settings
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25

# Use the data we already downloaded
config["background_paths"] = [
    "./audioset_16k",
    "./fma"
]

config["false_positive_validation_data_path"] = (
    "./validation_set_features.npy"
)

config["feature_data_files"] = {
    "ACAV100M_sample":
        "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}

# Save the configuration
with open("/content/my_model.yaml", "w") as file:
    yaml.dump(config, file)

print("✅ Configuration saved!")
print("Wake word:", config["target_phrase"])
print("Model name:", config["model_name"])

✅ Configuration saved!
Wake word: ['hey raizen']
Model name: hey_raizen


In [17]:
!ls -ld /content/audioset_16k /content/fma

ls: cannot access '/content/fma': No such file or directory
drwxr-xr-x 2 root root 69632 Aug 20 15:32 /content/audioset_16k


In [18]:
# Use only the background data that actually exists
config["background_paths"] = [
    "/content/audioset_16k"
]

# Save the corrected configuration
with open("/content/my_model.yaml", "w") as file:
    yaml.dump(config, file)

print("✅ Background path fixed!")
print(config["background_paths"])

✅ Background path fixed!
['/content/audioset_16k']


# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [5]:
!python /content/openwakeword/openwakeword/train.py \
  --training_config /content/openwakeword/examples/custom_model.yml \
  --generate_clips

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/beartype/_check/forward/fwdresolve.py", line 733, in _resolve_func_scope_forward_hint
    hint_resolved = eval(hint, decor_meta.func_wrappee_scope_forward)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
AttributeError: module 'onnxscript.values' has no attribute 'ParamSchema'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/openwakeword/openwakeword/train.py", line 4, in <module>
    import torchmetrics
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/__init__.py", line 14, in <module>
    from torchmetrics import functional  # noqa: E402
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchmetrics/functional/__init__.py", line 14, in <module>
    from torchmetrics.functional.audio._deprecated import _permutation_invari

In [6]:
!pip uninstall -y onnxscript
!pip install "onnxscript==0.1.0"

Found existing installation: onnxscript 0.7.1
Uninstalling onnxscript-0.7.1:
  Successfully uninstalled onnxscript-0.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 702.7/702.7 kB 11.9 MB/s eta 0:00:00


In [1]:
import torch
import torchaudio
import onnxscript

print("Torch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("ONNXScript:", onnxscript.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.4.1+cu121
Torchaudio: 2.4.1+cu121
ONNXScript: 0.1.0
CUDA available: True


In [20]:
!pip uninstall -y torch torchaudio torchvision
!pip install torch==2.4.1 torchaudio==2.4.1 torchvision==0.19.1

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 MB 789.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
!nvidia-smi

Thu Aug 20 16:09:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!ls -lh /content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!ls -lh /content/validation_set_features.npy
!ls -d /content/audioset_16k
!ls -d /content/openwakeword

-rw-r--r-- 1 root root 17G Aug 20 15:42 /content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
-rw-r--r-- 1 root root 177M Aug 20 15:42 /content/validation_set_features.npy
/content/audioset_16k
/content/openwakeword


In [ ]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

In [ ]:
# Step 3: Train model

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!